# SonaWills — Free Kaggle GPU Worker

This notebook runs the existing Wan2.2 TI2V-5B generation backend on Kaggle's free GPU session. It does not add SonaWills features. Keep this notebook running while you generate videos.

The notebook uses both available Tesla T4 GPUs with Wan2.2's official FSDP multi-GPU inference path. Wan2.2 documents TI2V-5B multi-GPU inference with FSDP/DeepSpeed Ulysses; the single-GPU 720p path needs at least 24 GB VRAM, so the two-T4 setup is used here.


In [ ]:
import os, sys, subprocess, time, json, base64, threading, re, pathlib
from pathlib import Path

ROOT = Path('/kaggle/working/sonawills')
WAN_DIR = Path('/kaggle/working/Wan2.2')
MODEL_DIR = Path('/kaggle/working/Wan2.2-TI2V-5B')
print('Kaggle GPU worker setup starting...')
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader']).decode())


In [ ]:
if not (ROOT / 'server/gpu/kaggle_worker.py').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/chineduwilliams739-commits/sonawills.git',str(ROOT)], check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only'], check=False)
if not WAN_DIR.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Wan-Video/Wan2.2.git',str(WAN_DIR)], check=True)
print('Repositories ready.')


In [ ]:
# Install Wan2.2 dependencies without flash-attn. Wan2.2 falls back to PyTorch scaled-dot-product attention when flash-attn is unavailable.
packages = [
    'opencv-python>=4.9.0.80', 'diffusers>=0.31.0', 'transformers>=4.49.0,<=4.51.3',
    'tokenizers>=0.20.3', 'accelerate>=1.1.1', 'tqdm', 'imageio[ffmpeg]', 'easydict',
    'ftfy', 'dashscope', 'imageio-ffmpeg', 'numpy>=1.23.5,<2', 'sentencepiece', 'fastapi', 'uvicorn', 'python-multipart', 'requests'
]
subprocess.run([sys.executable,'-m','pip','install','-q','-U',*packages], check=True)
print('Dependencies installed.')


In [ ]:
from huggingface_hub import snapshot_download
if not (MODEL_DIR / '.ready').exists():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id='Wan-AI/Wan2.2-TI2V-5B', local_dir=str(MODEL_DIR))
    (MODEL_DIR / '.ready').write_text('ready\n')
print('Wan2.2 TI2V-5B model is ready.')


In [ ]:
# Install Cloudflare's free Quick Tunnel client. It creates a temporary public URL without a paid account.
cloudflared = Path('/kaggle/working/cloudflared')
if not cloudflared.exists():
    subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O',str(cloudflared)], check=True)
    cloudflared.chmod(0o755)
print('cloudflared ready.')


In [ ]:
import os
os.environ['PYTHONPATH'] = f"{ROOT}:{WAN_DIR}:" + os.environ.get('PYTHONPATH','')
os.environ['WAN_DIR'] = str(WAN_DIR)
os.environ['MODEL_DIR'] = str(MODEL_DIR)
os.environ['SONAWILLS_ROOT'] = '/kaggle/working/sonawills-worker'

# Start the SonaWills API.
api_log = open('/kaggle/working/sonawills-api.log','w')
api = subprocess.Popen([sys.executable,'-m','uvicorn','server.gpu.kaggle_worker:app','--host','0.0.0.0','--port','8000'], cwd=str(ROOT), env=os.environ.copy(), stdout=api_log, stderr=subprocess.STDOUT)
time.sleep(5)
print('API process:', api.poll() if api.poll() is not None else 'running')


In [ ]:
# Start a temporary public HTTPS tunnel and capture its URL.
tunnel_log = open('/kaggle/working/sonawills-tunnel.log','w')
tunnel = subprocess.Popen([str(cloudflared),'tunnel','--no-autoupdate','--url','http://127.0.0.1:8000'], stdout=tunnel_log, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(60):
    time.sleep(2)
    text = Path('/kaggle/working/sonawills-tunnel.log').read_text(errors='ignore')
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
    if m:
        public_url = m.group(0)
        break
if not public_url:
    print(Path('/kaggle/working/sonawills-tunnel.log').read_text(errors='ignore')[-4000:])
    raise RuntimeError('Cloudflare Quick Tunnel URL was not created.')
print('SonaWills GPU URL:', public_url)
print('Keep this Kaggle notebook session running while generating videos.')


In [ ]:
# Sync the temporary GPU URL to the GitHub Pages frontend when a GitHub token is available.
# Add a Kaggle secret named SONAWILLS_GITHUB_TOKEN containing a fine-grained GitHub token
# with Contents: Read and write permission for chineduwilliams739-commits/sonawills.
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('SONAWILLS_GITHUB_TOKEN')
except Exception:
    token = None

if token:
    import requests
    api_url = 'https://api.github.com/repos/chineduwilliams739-commits/sonawills/contents/public/gpu-config.json'
    headers = {'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'X-GitHub-Api-Version':'2022-11-28'}
    current = requests.get(api_url, headers=headers, timeout=30)
    current.raise_for_status()
    current_json = current.json()
    config = {'url': public_url, 'provider':'kaggle', 'model':'Wan2.2-TI2V-5B', 'status':'online', 'updatedAt':time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
    payload = base64.b64encode((json.dumps(config, indent=2)+'\n').encode()).decode()
    update = requests.put(api_url, headers=headers, json={'message':'chore: sync free Kaggle GPU endpoint','content':payload,'sha':current_json['sha'],'branch':'main'}, timeout=30)
    update.raise_for_status()
    print('GitHub Pages GPU configuration updated.')
else:
    print('SONAWILLS_GITHUB_TOKEN was not available. The API is still running at:')
    print(public_url)
    print('Add the token secret and rerun this cell to sync the URL automatically.')


In [ ]:
import requests
health = requests.get(public_url + '/health', timeout=30)
print('Health:', health.status_code, health.text)
assert health.ok
print('SonaWills free GPU worker is online.')


## Keep this session running

Kaggle's free GPU is quota-based and the public Quick Tunnel is temporary. If the Kaggle session stops, the temporary URL stops too. Start this notebook again to bring the existing SonaWills GPU backend back online. Kaggle documents a weekly GPU quota, typically 30 hours or sometimes higher depending on demand.


In [ ]:
# Optional keep-alive cell. Leave it running during a generation session.
while True:
    time.sleep(60)
